You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [3]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [11]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Title: The Future of Artificial Intelligence: Trends, Players, and News

Introduction:
In recent years, Artificial Intelligence (AI) has rapidly evolved and is now being integrated into various industries, changing the way we live and work. This blog article will explore the latest trends, key players, and noteworthy news in the field of AI, providing valuable insights for our readers.

Key Points:
1. Latest Trends in Artificial Intelligence:
- Machine Learning: The use of algorithms and statistical models t

I now can give a great answer

Final Answer:
# The Future of Artificial Intelligence: Trends, Players, and News

## Introduction
Artificial Intelligence (AI) has become a transformative force in various industries, revolutionizing the way we interact with technology and shaping the future of work and society. In this blog post, we will delve into the latest trends, key players, and noteworthy news in the field of AI, providing valuable insights for professionals, business leaders, and individuals interested in the evolving landscape of artificial intelligence.

## Latest Trends in Artificial Intelligence
One of the most prominent trends in AI is the widespread adoption of Machine Learning, a subset of AI that enables machines to learn from data and make decisions without explicit programming. Algorithms and statistical models are being used to power a variety of applications, from recommendation systems to autonomous vehicles. Natural Language Processing (NLP) is another key trend, all

- Display the results of your execution as markdown in the notebook.

In [12]:
from IPython.display import Markdown
Markdown(result)

# The Future of Artificial Intelligence: Trends, Players, and News

## Introduction
Artificial Intelligence (AI) has become a transformative force in various industries, revolutionizing the way we interact with technology and shaping the future of work and society. In this blog post, we will delve into the latest trends, key players, and noteworthy news in the field of AI, providing valuable insights for professionals, business leaders, and individuals interested in the evolving landscape of artificial intelligence.

## Latest Trends in Artificial Intelligence
One of the most prominent trends in AI is the widespread adoption of Machine Learning, a subset of AI that enables machines to learn from data and make decisions without explicit programming. Algorithms and statistical models are being used to power a variety of applications, from recommendation systems to autonomous vehicles. Natural Language Processing (NLP) is another key trend, allowing machines to understand, interpret, and generate human language. This technology is being leveraged in virtual assistants, chatbots, and language translation tools. Additionally, Robotics is playing a crucial role in AI advancements, with robots being integrated with AI capabilities to perform tasks autonomously and collaborate with humans in various settings.

## Key Players in Artificial Intelligence
Several tech giants are at the forefront of AI research and development, shaping the future of the industry. Google, with projects like Google Brain and DeepMind, is leading the way in AI innovation. IBM is known for its AI platform Watson, which is utilized across industries for data analysis and decision-making. Microsoft is heavily investing in AI technologies such as Azure AI and Cognitive Services, while Amazon is leveraging AI in its e-commerce platform, cloud services, and virtual assistant Alexa. These key players are driving the evolution of AI and exploring new frontiers in technology.

## Noteworthy News in Artificial Intelligence
As AI continues to advance, there are important discussions surrounding ethics in AI development and deployment. The ethical implications of AI technologies, such as bias in algorithms and data privacy concerns, are being debated within the industry and among policymakers. In the healthcare sector, AI is being used for diagnosing diseases, creating personalized treatment plans, and accelerating drug discovery processes. In finance, AI-powered tools are enabling fraud detection, risk assessment, and automated trading, revolutionizing traditional practices in the industry.

## Conclusion
Artificial Intelligence is a rapidly evolving field with far-reaching implications for society, business, and technology. By staying informed on the latest trends, key players, and news in AI, professionals and individuals can navigate the complex landscape of artificial intelligence and harness its potential for innovation and growth. As we continue to explore the possibilities of AI, it is essential to consider the ethical and practical implications of these technologies to ensure a responsible and inclusive future.

*Stay updated on the latest developments in Artificial Intelligence by subscribing to our newsletter for regular insights and analysis on AI trends and innovations.*

By incorporating SEO keywords and engaging content, this blog post aims to provide valuable information on Artificial Intelligence, catering to the interests of our target audience and fostering a deeper understanding of the evolving AI landscape.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [13]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on YOUR TOPIC HERE.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Content Plan Document

Topic: The Impact of Artificial Intelligence on Marketing Strategies

1. Latest Trends, Key Players, and Noteworthy News
- Latest Trends: 
    - Use of AI in personalized marketing campaigns
    - AI-powered chatbots for customer service
    - Predictive analytics for targeted advertising
- Key Players: 
    - Google with its AI-powered tools like Google Ads and Google Analytics
    - Salesforce with AI-driven CRM solutions
    - IBM with Watson AI for marketing automation
- Noteworthy News:
 

I now can give a great answer

Final Answer: 

# The Impact of Artificial Intelligence on Marketing Strategies

In today's fast-paced digital world, the influence of artificial intelligence on marketing strategies is becoming increasingly prominent. The integration of AI in personalized marketing campaigns, the use of AI-powered chatbots for customer service, and the implementation of predictive analytics for targeted advertising are just a few examples of how this technology is reshaping the way businesses connect with their target audience. Major industry players like Google, Salesforce, and IBM are at the forefront of this revolution, demonstrating the vast potential AI holds in transforming marketing practices.

Recent research has indicated that AI has the potential to boost marketing ROI by 20%, highlighting the competitive advantage that businesses can gain by adopting this technology. From enhancing customer engagement to optimizing advertising strategies, AI is proving to be a

In [14]:
Markdown(result)

# The Impact of Artificial Intelligence on Marketing Strategies

In today's fast-paced digital world, the influence of artificial intelligence on marketing strategies is becoming increasingly prominent. The integration of AI in personalized marketing campaigns, the use of AI-powered chatbots for customer service, and the implementation of predictive analytics for targeted advertising are just a few examples of how this technology is reshaping the way businesses connect with their target audience. Major industry players like Google, Salesforce, and IBM are at the forefront of this revolution, demonstrating the vast potential AI holds in transforming marketing practices.

Recent research has indicated that AI has the potential to boost marketing ROI by 20%, highlighting the competitive advantage that businesses can gain by adopting this technology. From enhancing customer engagement to optimizing advertising strategies, AI is proving to be a game-changer for marketing professionals seeking to elevate their approaches and for business owners aiming to maximize their returns. By incorporating data-driven techniques and keeping abreast of the latest AI developments, businesses can effectively engage their target audience and accomplish their marketing objectives.

## AI Applications in Personalized Marketing

One of the primary benefits of utilizing AI in marketing strategies is its capacity to personalize the customer experience. Through the analysis of extensive data sets, AI can craft customized marketing initiatives that resonate with individual preferences and behaviors. This level of personalization not only boosts customer engagement but also enhances conversion rates and fosters customer loyalty. With the aid of AI-driven tools such as Google Ads and Salesforce's CRM solutions, businesses can deliver precise messaging that directly addresses the needs and interests of their audience.

## Benefits of AI-Powered Chatbots for Customer Service

AI-powered chatbots have emerged as a valuable resource for businesses seeking to streamline their customer service operations. These intelligent bots are adept at handling customer inquiries, providing immediate support, and even recommending products or services based on customer interactions. By leveraging AI chatbots, businesses can enhance response times, reduce operational expenses, and offer a seamless customer experience around the clock. With IBM's Watson AI leading the charge in marketing automation, businesses can strengthen their customer service capabilities and cultivate stronger connections with their audience.

## Using Predictive Analytics for Targeted Advertising

Another area where AI is revolutionizing marketing strategies is predictive analytics. By scrutinizing historical data and trends, AI algorithms can forecast future outcomes and behaviors, enabling businesses to optimize their advertising endeavors for maximum impact. Equipped with AI-powered tools like Google Analytics, businesses can glean invaluable insights into customer behavior, preferences, and trends, empowering them to craft targeted advertising campaigns that resonate with their audience. This data-centric approach not only enhances ROI but also positions businesses ahead of the competition in an ever-evolving market landscape.

In summary, the significance of artificial intelligence in marketing strategies cannot be overstated. As AI continues to progress and evolve, businesses that embrace this technology will undoubtedly secure a competitive advantage and achieve heightened success in their marketing initiatives. By remaining abreast of the latest AI trends, leveraging AI tools for personalized marketing campaigns, utilizing chatbots for customer service, and harnessing predictive analytics for targeted advertising, businesses can refine their strategies, effectively engage their target audience, and ultimately drive superior outcomes. It is time for businesses to explore the boundless potential of AI in marketing and unlock the transformative power of this technology.

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).